# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Notebook chạy được trên **Colab (T4)** hoặc **Kaggle (GPU T4)**. Mã dùng chung nằm trong `lab10.py`; mỗi ô ở đây gọi hàm tương ứng với một bước của `GUIDE.md`.

## 0. Môi trường
- **Colab:** Runtime → Change runtime type → T4 GPU. Secrets 🔑: `KAGGLE_USERNAME`, `KAGGLE_KEY` (bật *Notebook access*).
- **Kaggle:** Settings → Accelerator **GPU T4 x2**, **Internet on**; *Add Input* → dataset `road-damage-dataset-potholes-cracks-and-manholes`.

`work/` (adapter, dự đoán) được đặt ở nơi không mất khi session bị ngắt: Google Drive (Colab) hoặc `/kaggle/working` (Kaggle).

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
REPO = 'https://github.com/long2110d/K4-Track-4-Day10-VisionLanguageModel.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, PERSIST = '/content/lab10', '/content/drive/MyDrive/lab10_work'
elif IN_KAGGLE:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # T4 x2: chỉ dùng 1 GPU
    BASE, PERSIST = '/kaggle/working/lab10', '/kaggle/working/lab10_work'
else:
    BASE, PERSIST = os.getcwd(), None

if not os.path.exists(BASE):
    subprocess.run(['git', 'clone', REPO, BASE], check=True)
else:
    subprocess.run(['git', '-C', BASE, 'pull'], check=False)
os.chdir(BASE)
if PERSIST:
    os.makedirs(PERSIST, exist_ok=True)
    if not os.path.exists('work'):
        os.symlink(PERSIST, 'work')
sys.path.insert(0, BASE)
print('thư mục làm việc:', os.getcwd())

In [ ]:
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
# Kaggle API key: Colab Secrets / Kaggle Secrets. KHÔNG viết khoá vào notebook.
if IN_COLAB:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
import glob
if os.path.exists('data/splits/meta.json'):
    print('đã có data/, bỏ qua')
elif IN_KAGGLE and glob.glob('/kaggle/input/**/labels-YOLO', recursive=True):
    !python scripts/prepare_data.py --source /kaggle/input      # dataset gắn qua Add Input
else:
    !python scripts/prepare_data.py --kaggle
!cat data/splits/meta.json | head -5

In [ ]:
import json, lab10
from lab10 import *
sessions = split_stats()
train_rows, test_rows = load_split('train'), load_split('test')
show_grid([draw_boxes(e, title=e['image_id'][:40]) for e in train_rows[::240][:6]],
          path='report/figures/train_samples.jpg')

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
e = next(r for r in train_rows if len(r['objects']) >= 2)
ans = objects_to_answer(e['objects'])
print(ans)
# quy ngược về pixel và đối chiếu với nhãn YOLO gốc (sai số chỉ do làm tròn)
from scripts.score import _gt_boxes
gt = sorted((g['bbox'] for g in _gt_boxes(e)), key=lambda b: (round(b[1]), round(b[0])))
for b, g in zip(json.loads(ans), gt):
    print(b['label'], [round(v, 1) for v in norm1000_to_pixel(b['bbox_2d'], 640, 360)],
          [round(v, 1) for v in norm1000_to_pixel(g, 640, 360)])
print(PROMPT)
draw_boxes(e, raw=ans, title='xanh = GT, do = JSON da quy doi')

## 3. E0: zero-shot

In [ ]:
import torch
model, processor = load_model()
print({k: f'{v/1e6:.0f}M' for k, v in count_blocks(model).items()})
val40 = load_split('val')[:40]
predict(model, processor, val40, 'work/e0_val40.jsonl')         # gỡ lỗi nhanh
score('work/e0_val40.jsonl', 'E0-val40', 'work/e0_val40.json', split='val')

In [ ]:
predict(model, processor, test_rows, 'work/e0.jsonl')
score('work/e0.jsonl', 'E0', 'report/evidence/e0.json')
for r in lab10.read_jsonl('work/e0.jsonl')[:5]:
    print(repr(r['raw'][:300]), '\n')

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
n_img, ans = check_example(processor, train_rows[0])
assert n_img == 220, n_img
batch = make_collate(processor)(train_rows[:2])
print({k: tuple(v.shape) for k, v in batch.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3
Mỗi thí nghiệm nạp lại model gốc rồi gọi `apply_setup`. Đặt `SETUP` rồi chạy các ô 5 → 7; làm lần lượt E1, E2, E3.

In [ ]:
import gc, yaml
SETUP = 'E1'            # đổi thành 'E2', 'E3'
cfg = yaml.safe_load(open(f'configs/{SETUP.lower()}.yaml'))
model = None; gc.collect(); torch.cuda.empty_cache()
model, processor = load_model()
model, n_matched, n_train = apply_setup(model, SETUP, r=cfg['lora']['r'], alpha=cfg['lora']['alpha'],
                                        dropout=cfg['lora']['dropout'])

## 6. Vòng huấn luyện

In [ ]:
OUT = f'work/adapter_{SETUP.lower()}'
run = train(model, processor, OUT, SETUP, lr=cfg['lr'], batch_size=cfg['batch_size'],
            grad_accum=cfg['grad_accum'], max_steps=cfg['max_steps'], seed=cfg['seed'])

## 7. Suy luận và chấm E1–E3

In [ ]:
name = SETUP.lower()
predict(model, processor, val40, f'work/{name}_val40.jsonl')
score(f'work/{name}_val40.jsonl', f'{SETUP}-val40', f'work/{name}_val40.json', split='val')

In [ ]:
# Chấm test một lần cho mỗi setup. Nếu session bị ngắt: model, processor = load_model(OUT) rồi chạy lại ô này.
predict(model, processor, test_rows, f'work/{name}.jsonl')
score(f'work/{name}.jsonl', SETUP, f'report/evidence/{name}.json', run_json=f'{OUT}/run.json')
!cp -r report work/   # sao lưu evidence sang Drive / output

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
E4: đổi **rank** trên setup tốt nhất của E1–E3 (r=4 và r=64, alpha = 2r). E5: bỏ qua.

In [ ]:
for k in ('e0', 'e1', 'e2', 'e3'):
    d = json.load(open(f'report/evidence/{k}.json'))
    print(k, round(d['micro']['f1'], 3), [round(x, 3) for x in d['f1_ci95']])
BEST = 'E1'             # điền setup có F1 tốt nhất

In [ ]:
for r in (4, 64):
    model = None; gc.collect(); torch.cuda.empty_cache()
    model, processor = load_model()
    model, _, _ = apply_setup(model, BEST, r=r, alpha=2 * r)
    out = f'work/adapter_e4_rank_{r}'
    train(model, processor, out, BEST, max_steps=300)
    predict(model, processor, test_rows, f'work/e4_rank_{r}.jsonl')
    score(f'work/e4_rank_{r}.jsonl', f'E4-rank{r}', f'report/evidence/e4_rank_{r}.json', run_json=f'{out}/run.json')
!cp -r report work/

### Ảnh minh hoạ đúng/sai (box đúng màu xanh lá, dự đoán màu đỏ)

In [ ]:
preds = {r['image_id']: r['raw'] for r in lab10.read_jsonl(f'work/{BEST.lower()}.jsonl')}
f1 = per_image_f1(test_rows, preds)
has_obj = [e for e in test_rows if e['objects']]
ranked = sorted(has_obj, key=lambda e: f1[e['image_id']]['micro']['f1'])
picks = ranked[-3:] + ranked[:3]
imgs = [draw_boxes(e, preds[e['image_id']], title=f"{BEST} F1={f1[e['image_id']]['micro']['f1']:.2f} {e['image_id'][:28]}") for e in picks]
show_grid(imgs, path=f'report/figures/{BEST.lower()}_good_bad.jpg')

In [ ]:
# Lỗi theo lớp: mỗi lớp một ảnh sai
for c in ('pothole', 'crack', 'manhole'):
    bad = [e for e in ranked if any(o['label'] == c for o in e['objects'])][:3]
    show_grid([draw_boxes(e, preds[e['image_id']], title=f'{c} {e["image_id"][:30]}') for e in bad],
              path=f'report/figures/errors_{c}.jpg')
!cp -r report work/